In [0]:
"""
===============================================================
Module: Superstore Gold facts ETL pipeline orchestrator

Purpose:
    This module orchestrates the ETL process for fact tables in the Gold layer of the Superstore 
    Medallion Architecture. It reads data from the Silver layer, processes the data by preparing 
    fact columns, merging new data into the Gold fact tables, handling incremental data processing, 
    collecting metrics, and optimizing Gold tables for query performance.

Key Features:
1. Incremental Fact Table Processing:
    - Reads incremental data from the Silver layer, processes new rows, and merges them into the Gold layer.
    - Tracks new, updated, and unchanged rows efficiently, ensuring minimal reprocessing.

2. Data Preparation:
    - Prepares data for Gold tables by adding necessary columns and ensuring the correct schema.
    - Allows for dynamic handling of different fact tables using configuration.

3. Gold Table Management:
    - Creates Gold tables if they do not exist, ensuring that the pipeline is idempotent.
    - Merges data into the Gold fact tables, handling updates and inserts appropriately.
    - OPTIMIZE and VACUUM are left to Unity Catalog Predictive Optimization; this pipeline runs neither.

4. Error Handling & Logging:
    - Provides detailed logging and structured observability using master_run_id and layer_run_id.
    - Proper error handling ensures that any issues in the pipeline are logged with detailed error messages.

5. Metrics & Observability:
    - Collects and logs key metrics such as rows read, rows inserted, and rows updated for each fact table.
    - Provides insights into the performance and progress of each ETL pipeline.

6. Production-Ready:
    - Designed to process large volumes of data in a distributed environment, suitable for production-grade ETL.
    - Runs efficiently on Databricks or any Spark-enabled platform, ensuring scalability.

Best Practices:
- Ensure that the `superstore_gold_facts_config.yaml` file is configured correctly to map Silver tables to Gold tables.
- Regularly monitor the performance metrics and track the status of fact table processing.
===============================================================
"""
# -------------------------------
# Core Python + Utility Imports
# -------------------------------
# Standard libraries for Spark session, environment handling, timestamps, YAML parsing, and dynamic module reload
from pyspark.sql import SparkSession  # Spark entry point for distributed processing
import os, sys, time, yaml, importlib  # OS/system utilities, timing, YAML config parsing, module reload
from datetime import datetime  # Timestamp utilities for logging and audit
import uuid  # Unique identifier generation for pipeline runs

# -------------------------------
# Spark Session Initialization
# -------------------------------
# Initialize or reuse Spark session
spark = SparkSession.builder.getOrCreate()

# Ensure consistent timestamp handling across pipeline (UTC standardization)
spark.conf.set("spark.sql.session.timeZone", "UTC")


# -------------------------------
# Runtime Environment Configuration
# -------------------------------
# Fetch runtime environment variables from Databricks job context
env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

# Set environment variables for downstream modules (logging, config, observability)
os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version


# -------------------------------
# Add Shared Code Paths
# -------------------------------
# this notebook lives at <bundle_root>/superstore_orchestrator/layer_orchestrator/<name>
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
# Path for shared utilities (logging, config management, platform helpers)
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")
# Path for Gold layer domain-specific framework (SCD2 logic, merges, metrics, etc.)
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_gold/core")  

# -------------------------------
# Core Framework Imports
# -------------------------------
# Logging and event tracking utilities
from superstore_logger import get_superstore_logger, log_event

# Execution context helper (master + layer run tracking)
from superstore_orchestrator_execution_context import get_execution_context

# Platform constants (standardized layer naming)
from superstore_platform_constants import GOLD_LAYER

# Platform config utilities (catalog/schema/table resolution helpers)
from superstore_platform_config import (
    table,
    get_silver_schema,
    get_gold_schema,
    get_metrics_schema
)
from superstore_backfill_utils import (
    get_backfill_config,
    validate_backfill_impact
)


# -------------------------------
# Execution Context Initialization
# -------------------------------
# Retrieve pipeline execution identifiers for traceability and observability
execution_context = get_execution_context(dbutils, allow_standalone=True)
master_run_id = execution_context["master_run_id"]
layer_run_id = execution_context["layer_run_id"]

# -------------------------------
# Load Fact Table Configuration
# -------------------------------
# Load YAML configuration defining fact tables, keys, and processing rules
config_path = f"{BUNDLE_ROOT}/configs/superstore_gold_config/superstore_gold_facts_config.yaml"  
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
    
# Extract fact table configuration block
fact_table_configs = config["fact_table_configs"]


# -------------------------------
# Import Gold Facts Framework
# -------------------------------
# Load reusable fact processing framework (MERGE, UPSERT, metrics, optimization)
# Reload ensures latest code changes are applied in notebook execution (dev mode)
import superstore_gold_facts_framework
importlib.reload(superstore_gold_facts_framework)

from superstore_gold_facts_framework import (
    read_silver_table,              # Reads Silver fact source data
    prepare_fact_columns,           # Prepares derived and hash columns for fact processing
    create_gold_table_if_not_exists,# Ensures Gold fact table exists
    merge_fact_into_gold,           # Performs UPSERT/MERGE into Gold fact table
    collect_fact_metrics            # Captures fact layer metrics (volume, performance, skew)
)

# -------------------------------
# Logger Initialization
# -------------------------------
# Initialize orchestrator logger for Gold facts pipeline
logger_gold_facts_orchestrator = get_superstore_logger("superstore_gold_facts_orchestrator")

# Log pipeline start event (audit + observability)
log_event(
    logger_gold_facts_orchestrator,
    "INFO",
    "Initiating Superstore Gold layer ETL facts orchestrator",
    master_run_id=master_run_id,
    layer_run_id=layer_run_id,
    layer=GOLD_LAYER
)


# Get backfill_config
backfill_config = get_backfill_config(dbutils, allow_full_refresh=False)

log_event(
    logger_gold_facts_orchestrator,
    "INFO",
    "Gold fact run configuration loaded",
    run_mode=backfill_config["mode"],
    dry_run=backfill_config["dry_run"],
    is_windowed=backfill_config["is_windowed"],
    master_run_id=master_run_id,
    layer_run_id=layer_run_id,
    layer=GOLD_LAYER
)

# -------------------------------
# Dry run
# -------------------------------
# Report what this run would touch, then exit before any write. Every task
# that writes must honour dry_run: a preview that writes to some layers while
# the DAG reports success is worse than having no flag, because it creates
# false confidence exactly when an operator is being careful.
if backfill_config["dry_run"]:
    for _cfg_name, cfg in fact_table_configs.items():
        source_tbl = table(get_silver_schema(), cfg["silver_tbl"])
        impact = validate_backfill_impact(
            spark,
            source_tbl,
            backfill_config,
            date_column="ingestion_date"
        )
        log_event(
            logger_gold_facts_orchestrator,
            "INFO",
            f"Dry-run impact for {source_tbl}",
            run_mode=backfill_config["mode"],
            impact=impact,
            master_run_id=master_run_id,
            layer_run_id=layer_run_id,
            layer=GOLD_LAYER
        )
    dbutils.notebook.exit(
        "DRY_RUN_COMPLETED run_mode=" + backfill_config["mode"] + " layer=gold_facts (no data written)"
    )

# -----------------------------
# Master Orchestrator Function
# -----------------------------
def run_all_fact_pipelines(spark, master_run_id: str):
    log_event(
        logger_gold_facts_orchestrator,
        "INFO",
        "Starting Gold fact ETL for all configured tables",
        total_tables=len(fact_table_configs),
        master_run_id=master_run_id,
        layer_run_id=layer_run_id,
        layer=GOLD_LAYER
    )

    for table_name, cfg in fact_table_configs.items():

        # -----------------------------
        # Dynamically resolve full catalog.schema.table
        # -----------------------------
        silver_tbl = table(get_silver_schema(), cfg["silver_tbl"])
        gold_tbl = table(get_gold_schema(), cfg["gold_tbl"])
        metrics_table = table(get_metrics_schema(), cfg["metrics_table"])

        layer_name = cfg["layer_name"]
        table_type = cfg["table_type"]
        required_columns = cfg["required_columns"]
        natural_keys = cfg["natural_keys"]
        hash_column = cfg["hash_column"]
        meta_columns= cfg["meta_columns"]
    

        start_ts = spark.sql("SELECT current_timestamp() as ts").first()["ts"]
        start_time = time.time()
        # run_status = "SUCCESS"
        # notes = ""
        has_incremental_rows = False

        try:
            log_event(
                logger_gold_facts_orchestrator,
                "INFO",
                f"Starting fact pipeline for table: {gold_tbl}",
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )

            # Step 1: Read Silver table
            df_silver = read_silver_table(
                spark,
                silver_table=silver_tbl,
                gold_table=gold_tbl,
                backfill_config=backfill_config,  # backfill
                required_columns=required_columns,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )

            # -----------------------------------------
            # CASE A: Silver table does NOT exist
            # -----------------------------------------
            if df_silver is None:
                log_event(
                    logger_gold_facts_orchestrator,
                    "INFO", 
                    f"No {silver_tbl} table found in Silver for {gold_tbl} ")

                collect_fact_metrics(
                    spark,
                    df=df_silver,
                    metrics_table=metrics_table,
                    merge_metrics=None,
                    is_initial_load=None,
                    layer_name=layer_name,
                    silver_table=silver_tbl,
                    gold_table=gold_tbl,
                    table_type=table_type,
                    start_ts=start_ts,
                    end_ts=start_ts,
                    duration_secs=0,
                    # run_status="skipped",
                    # notes="Silver table not found",
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER,
                    backfill_config=backfill_config
                )
                continue


            # -----------------------------------------
            # CASE B: Silver exists but EMPTY
            # -----------------------------------------
            has_incremental_rows = df_silver.limit(1).count() > 0
            if not has_incremental_rows:
                log_event(
                    logger_gold_facts_orchestrator,
                    "INFO",
                    f"No new rows found in  {silver_tbl} for processing; skipping",
                    table=gold_tbl,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )
                collect_fact_metrics(
                    spark,
                    df=df_silver,
                    metrics_table=metrics_table,
                    merge_metrics=None,
                    is_initial_load=None,
                    layer_name=layer_name,
                    silver_table=silver_tbl,
                    gold_table=gold_tbl,
                    table_type=table_type,
                    start_ts=start_ts,
                    end_ts=start_ts,
                    duration_secs=0,
                    # run_status="skipped",
                    # notes="No new rows or files in silver to process",
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER,
                    backfill_config=backfill_config
                )
                # continue

            if has_incremental_rows:
                # Step 2: Prepare fact columns
                df_prepared = prepare_fact_columns(
                    df_silver,
                    entity_columns=required_columns,
                    meta_columns=meta_columns,
                    hash_column=hash_column,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )

                # Step 3: Create Gold table if missing
                create_gold_table_if_not_exists(
                    df_prepared,
                    gold_tbl,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )


                # ---------------------------------------------------------
                # Detect INITIAL LOAD condition for fact Gold table 
                # ---------------------------------------------------------
                # is_initial_load → TRUE when the table exists but contains no data
                #
                # In this pipeline:
                # - Gold table is typically created earlier (even if empty)
                # - So initial load = first time data is being written into an empty table
                #
                # This flag is later used to:
                # - Set execution_mode = "INITIAL_LOAD"
                # - Differentiate first-time load vs incremental SCD2 processing
                # ---------------------------------------------------------
                is_initial_load = spark.table(gold_tbl).limit(1).count() == 0

                # Step 4: Merge fact into Gold
                merge_metrics_results=merge_fact_into_gold(
                    df_prepared,
                    gold_tbl,
                    natural_keys,
                    hash_column,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )

                # Extract values safely
                merge_metrics = merge_metrics_results.get("merge_metrics", {})
                
                end_ts = spark.sql("SELECT current_timestamp() as ts").first()["ts"]
                duration_secs = int(time.time() - start_time)

                # Step 5: Metrics
                collect_fact_metrics(
                    spark,
                    df=df_silver,
                    metrics_table=metrics_table,
                    merge_metrics=merge_metrics,
                    is_initial_load=is_initial_load,
                    layer_name=layer_name,
                    silver_table=silver_tbl,
                    gold_table=gold_tbl,
                    table_type=table_type,
                    start_ts=start_ts,
                    end_ts=end_ts,
                    duration_secs=duration_secs,
                    # run_status=run_status,
                    # notes=notes,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER,
                    backfill_config=backfill_config
                )

                log_event(
                    logger_gold_facts_orchestrator,
                    "INFO",
                    f"Completed fact pipeline for table: {gold_tbl}",
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )
                
        except Exception as e:
            run_status = "FAILED"
            notes = str(e)
            logger_gold_facts_orchestrator.error(f"Fact pipeline failed for {gold_tbl}: {e}")
            raise

    log_event(
        logger_gold_facts_orchestrator,
        "INFO",
        "Completed executing fact pipeline for all tables",
        master_run_id=master_run_id,
        layer_run_id=layer_run_id,
        layer=GOLD_LAYER
    )


# -------------------------------
# Execute
# -------------------------------
if __name__ == "__main__":
    run_all_fact_pipelines(spark, master_run_id)
